# Reasoning Models — Advanced

> **Section 01 — Topic 08 — advanced**

**Prerequisites:** `08-reasoning-models/foundations.ipynb`

**What you'll learn:**
- How o1/o3-style reasoning models use thinking tokens and extended internal monologue
- Patterns in reasoning traces: backtracking, verification, decomposition
- Tree of Thought with BFS and DFS search strategies
- Tool-augmented reasoning with Python execution
- Claude's extended thinking API and budget tokens

**What you'll build:**
- A Tree of Thought implementation with BFS/DFS
- A tool-augmented reasoning pipeline with Python execution
- Extended thinking with Claude's API

## Setup

In [ ]:
!pip install -q openai anthropic matplotlib numpy

In [ ]:
import os
import re
import json
import time
import subprocess
import textwrap
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from openai import OpenAI
import anthropic

openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "your-key-here"))
anthropic_client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY", "your-key-here"))

FAST_MODEL = "gpt-4o-mini"

def chat_openai(messages, model=FAST_MODEL, temperature=0.0, max_tokens=2048):
    response = openai_client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens,
    )
    return response.choices[0].message.content

def ask(prompt, **kwargs):
    return chat_openai([{"role": "user", "content": prompt}], **kwargs)

print("Clients initialized")

---
## 1. o1/o3-Style Reasoning — Thinking Tokens

OpenAI's o1 and o3 models represent a paradigm shift: instead of the model producing reasoning steps *in the visible output*, the model generates **thinking tokens** in a hidden chain of thought before producing the final answer.

### Key properties of thinking-token models:

1. **Extended internal monologue:** The model may generate thousands of tokens of reasoning internally
2. **Backtracking:** The model can realize a reasoning path is wrong and try another
3. **Verification:** The model checks its own work before committing to an answer
4. **Trained with RL:** The model is trained to produce better reasoning via reinforcement learning, not just supervised on human demonstrations

### How it works architecturally:

```
User prompt → [hidden thinking tokens] → Visible answer
              ↑                          ↑
              Many tokens of internal     Short, final answer
              reasoning (not shown to     (shown to user)
              user but billed)
```

In [ ]:
# Simulate o1-style reasoning with explicit thinking blocks
# (Since we can't see o1's actual thinking tokens, we simulate the pattern)

def simulate_thinking_tokens(question, thinking_budget=1000):
    """
    Simulate o1-style reasoning by explicitly generating
    a thinking phase followed by a final answer.
    """
    thinking_prompt = f"""You are solving a complex problem. First, think through it extensively.
Show your complete internal reasoning process, including:
- Initial approach
- Checking for errors
- Considering alternatives
- Verifying your answer

Problem: {question}

<thinking>"""
    
    # Generate thinking phase
    thinking = ask(thinking_prompt, max_tokens=thinking_budget, temperature=0.3)
    
    # Generate final answer based on thinking
    answer_prompt = f"""Based on the following reasoning, give a clear, concise final answer.

Problem: {question}

Reasoning:
{thinking}

Final answer (just the answer, nothing else):"""
    
    answer = ask(answer_prompt, temperature=0.0)
    
    return thinking, answer

# Test on a problem that benefits from extended thinking
hard_problem = (
    "In a room of 23 people, what is the probability that at least two people "
    "share a birthday? Assume 365 days in a year and uniform distribution. "
    "Give the answer as a percentage rounded to 1 decimal place."
)

thinking, answer = simulate_thinking_tokens(hard_problem)
print(f"THINKING ({len(thinking.split())} words):")
print(f"{thinking[:500]}...\n")
print(f"ANSWER: {answer}")
print(f"\nExpected: ~50.7%")

In [ ]:
# Compare thinking budget impact
budgets = [100, 250, 500, 1000, 2000]
problem = "What is the 15th number in the Fibonacci sequence (starting from F(1)=1, F(2)=1)?"
expected = "610"

print(f"Problem: {problem}")
print(f"Expected: {expected}\n")

budget_results = []
for budget in budgets:
    thinking, answer = simulate_thinking_tokens(problem, thinking_budget=budget)
    thinking_tokens = len(thinking.split())
    correct = expected in answer
    budget_results.append({
        'budget': budget,
        'thinking_words': thinking_tokens,
        'correct': correct,
        'answer': answer.strip()[:50]
    })
    print(f"  Budget={budget:4d}: {answer.strip()[:30]:<30} thinking={thinking_tokens:4d} words  {'CORRECT' if correct else 'WRONG'}")

---
## 2. Reasoning Traces — Analyzing Patterns

When we analyze the internal reasoning of models (whether visible CoT or internal thinking tokens), we find recurring patterns:

1. **Decomposition:** Breaking a problem into sub-problems
2. **Backtracking:** "Wait, that's not right. Let me try again..."
3. **Verification:** "Let me check: if x = 5, then 3(5) + 7 = 22. Yes, correct."
4. **Hypothesis testing:** "This could be A or B. If A, then... if B, then..."
5. **Analogy:** "This is similar to [known problem]..."

Let's build a tool to detect and classify these patterns.

In [ ]:
class ReasoningTraceAnalyzer:
    """Analyze reasoning traces for common patterns."""
    
    PATTERNS = {
        'decomposition': [
            r'(?i)(first|step \d|break.*down|sub-?problem|let.*start.*with)',
            r'(?i)(part \d|component|piece by piece)',
        ],
        'backtracking': [
            r'(?i)(wait|actually|no,|hmm|let me re|that.*wrong|mistake|correction)',
            r'(?i)(reconsider|back up|start over|on second thought)',
        ],
        'verification': [
            r'(?i)(let.*check|verify|confirm|plug.*back|double.*check|sanity check)',
            r'(?i)(does this make sense|is this right|check.*work)',
        ],
        'hypothesis': [
            r'(?i)(if.*then|suppose|assume|what if|consider.*case)',
            r'(?i)(hypothesis|possibility|could be|might be)',
        ],
        'analogy': [
            r'(?i)(similar to|like|analogous|reminds me|same as|compare)',
            r'(?i)(recall|remember|known.*that)',
        ],
    }
    
    def analyze(self, trace):
        """Count occurrences of each reasoning pattern."""
        results = {}
        for pattern_name, regexes in self.PATTERNS.items():
            count = 0
            examples = []
            for regex in regexes:
                matches = re.finditer(regex, trace)
                for m in matches:
                    count += 1
                    # Get surrounding context
                    start = max(0, m.start() - 20)
                    end = min(len(trace), m.end() + 40)
                    examples.append(f"...{trace[start:end]}...")
            results[pattern_name] = {
                'count': count,
                'examples': examples[:3],  # Keep top 3 examples
            }
        return results

# Generate a detailed reasoning trace
hard_problem = (
    "A farmer has a fox, a chicken, and a bag of grain. He must cross a river with a boat "
    "that can carry only himself and one item at a time. If left alone, the fox will eat the "
    "chicken, and the chicken will eat the grain. How can he get everything across safely?"
)

trace = ask(
    f"""Solve this problem, showing ALL your thinking process. Include moments where you 
reconsider, verify steps, and explore alternatives.

{hard_problem}""",
    max_tokens=1500,
    temperature=0.3
)

print("Reasoning trace:")
print(trace[:600])
print("...\n")

# Analyze patterns
analyzer = ReasoningTraceAnalyzer()
analysis = analyzer.analyze(trace)

print("Pattern analysis:")
for pattern, info in analysis.items():
    print(f"  {pattern}: {info['count']} occurrences")
    for ex in info['examples'][:1]:
        print(f"    Example: {ex}")

In [ ]:
# Visualize reasoning patterns across different problem types
problem_types = {
    "Math": "Solve: What is the integral of x^2 * e^x dx? Show your complete thought process.",
    "Logic": """If all Bloops are Razzies, and all Razzies are Lazzies, and some Lazzies are Tazzies,
    can we conclude that some Bloops are Tazzies? Show your complete thought process.""",
    "Planning": """You need to schedule 5 meetings in one day. Each is 1 hour. 
    Meeting A must be before B. C must be after D. B and D cannot be adjacent. 
    E must be either first or last. Find a valid schedule. Show your complete thought process.""",
}

pattern_data = defaultdict(list)
type_names = []

for problem_type, prompt in problem_types.items():
    type_names.append(problem_type)
    trace = ask(prompt, max_tokens=1500, temperature=0.3)
    analysis = analyzer.analyze(trace)
    for pattern, info in analysis.items():
        pattern_data[pattern].append(info['count'])

# Plot pattern distribution
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(type_names))
width = 0.15
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

for i, (pattern, counts) in enumerate(pattern_data.items()):
    ax.bar(x + i * width, counts, width, label=pattern, color=colors[i % len(colors)])

ax.set_xlabel('Problem Type')
ax.set_ylabel('Pattern Occurrences')
ax.set_title('Reasoning Patterns by Problem Type')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(type_names)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Structured Reasoning — Tree of Thought

Tree of Thought (Yao et al., 2023) structures reasoning as a **search problem** over a tree of partial solutions. Unlike linear CoT, ToT can:

- Explore multiple reasoning paths in parallel
- Evaluate and prune unpromising paths
- Backtrack to try different approaches

### Search strategies:
- **BFS (Breadth-First):** Explore all paths at depth $d$ before going to $d+1$. Good when you want to compare alternatives.
- **DFS (Depth-First):** Follow each path to completion before backtracking. Good when paths are independent.

In [ ]:
class TreeOfThought:
    """
    Tree of Thought implementation with BFS and DFS search.
    
    Each node is a partial solution (string).
    At each step, the model proposes extensions and evaluates them.
    """
    
    def __init__(self, model=FAST_MODEL, branching_factor=3, max_depth=4):
        self.model = model
        self.branching_factor = branching_factor
        self.max_depth = max_depth
        self.nodes_explored = 0
    
    def propose_thoughts(self, problem, current_state, n=3):
        """Generate n possible next reasoning steps."""
        prompt = f"""Problem: {problem}

Current reasoning so far:
{current_state if current_state else '(nothing yet)'}

Propose {n} different possible next reasoning steps. Each should be a distinct approach or idea.
Format: one step per line, numbered 1-{n}. Each step should be 1-2 sentences."""
        
        response = ask(prompt, temperature=0.7, max_tokens=500)
        
        # Parse numbered steps
        thoughts = []
        for line in response.strip().split('\n'):
            line = line.strip()
            # Remove numbering
            cleaned = re.sub(r'^\d+[\.\)\:]\s*', '', line)
            if cleaned and len(cleaned) > 10:
                thoughts.append(cleaned)
        
        return thoughts[:n]
    
    def evaluate_thought(self, problem, state):
        """Score a partial reasoning state (0-10)."""
        prompt = f"""Problem: {problem}

Current reasoning:
{state}

On a scale of 1-10, how promising is this reasoning path for solving the problem?
Consider: Is it heading in the right direction? Is it logically sound? Does it make progress?

Score (just the number):"""
        
        response = ask(prompt, temperature=0.0, max_tokens=10)
        numbers = re.findall(r'\d+', response)
        return int(numbers[0]) if numbers else 5
    
    def solve_bfs(self, problem, beam_width=3):
        """BFS: explore all branches at each depth, keep top beam_width."""
        self.nodes_explored = 0
        current_states = [""]  # Start with empty state
        
        for depth in range(self.max_depth):
            all_candidates = []
            
            for state in current_states:
                thoughts = self.propose_thoughts(problem, state, n=self.branching_factor)
                for thought in thoughts:
                    new_state = f"{state}\nStep {depth+1}: {thought}" if state else f"Step 1: {thought}"
                    score = self.evaluate_thought(problem, new_state)
                    all_candidates.append((new_state, score))
                    self.nodes_explored += 1
            
            # Keep top beam_width candidates
            all_candidates.sort(key=lambda x: x[1], reverse=True)
            current_states = [s for s, _ in all_candidates[:beam_width]]
            
            print(f"  Depth {depth+1}: explored {len(all_candidates)} nodes, kept top {beam_width}")
        
        # Get final answer from best state
        best_state = current_states[0]
        answer_prompt = f"""Problem: {problem}

Reasoning:
{best_state}

Based on this reasoning, what is the final answer?"""
        
        answer = ask(answer_prompt)
        return best_state, answer
    
    def solve_dfs(self, problem, pruning_threshold=4):
        """DFS: follow each path deeply, prune low-scoring paths."""
        self.nodes_explored = 0
        best_solution = None
        best_score = 0
        
        def dfs(state, depth):
            nonlocal best_solution, best_score
            
            if depth >= self.max_depth:
                score = self.evaluate_thought(problem, state)
                if score > best_score:
                    best_score = score
                    best_solution = state
                return
            
            thoughts = self.propose_thoughts(problem, state, n=self.branching_factor)
            
            for thought in thoughts:
                new_state = f"{state}\nStep {depth+1}: {thought}" if state else f"Step 1: {thought}"
                self.nodes_explored += 1
                
                score = self.evaluate_thought(problem, new_state)
                
                # Prune unpromising branches
                if score >= pruning_threshold:
                    dfs(new_state, depth + 1)
                else:
                    print(f"  Pruned at depth {depth+1} (score={score})")
        
        dfs("", 0)
        
        if best_solution:
            answer_prompt = f"""Problem: {problem}

Reasoning:
{best_solution}

Based on this reasoning, what is the final answer?"""
            answer = ask(answer_prompt)
            return best_solution, answer
        
        return None, "Could not find solution"

print("Tree of Thought implementation ready")

In [ ]:
# Test Tree of Thought with BFS
tot = TreeOfThought(branching_factor=3, max_depth=3)

problem = (
    "I have numbers 1, 5, 6, 10. Using each number exactly once and "
    "basic arithmetic (+, -, *, /), make 24."
)

print(f"Problem: {problem}")
print(f"\nBFS Search:")
reasoning, answer = tot.solve_bfs(problem, beam_width=2)
print(f"\nNodes explored: {tot.nodes_explored}")
print(f"Best reasoning path:\n{reasoning}")
print(f"\nAnswer: {answer}")

In [ ]:
# Test with DFS for comparison
print(f"\nDFS Search:")
tot_dfs = TreeOfThought(branching_factor=2, max_depth=3)
reasoning_dfs, answer_dfs = tot_dfs.solve_dfs(problem, pruning_threshold=5)
print(f"\nNodes explored: {tot_dfs.nodes_explored}")
print(f"Answer: {answer_dfs}")

---
## 4. Tool-Augmented Reasoning — Python Executor

A powerful enhancement: let the model **write and execute code** as part of its reasoning process. This addresses a fundamental limitation — LLMs are unreliable at arithmetic and symbolic manipulation, but they can write code that performs these operations perfectly.

The pattern:
1. Model reasons about the problem in natural language
2. When it needs precise computation, it writes Python code
3. The code executes and returns results
4. The model incorporates the results into continued reasoning

In [ ]:
import ast

def safe_execute_python(code, timeout=5):
    """
    Execute Python code safely with a timeout.
    Returns (output, error).
    """
    # Basic safety: disallow imports of dangerous modules
    forbidden = ['os', 'sys', 'subprocess', 'shutil', 'pathlib', '__import__']
    for f in forbidden:
        if f in code and f != 'import math':
            return None, f"Forbidden operation: {f}"
    
    try:
        # Execute in a restricted namespace
        namespace = {'__builtins__': {
            'range': range, 'len': len, 'sum': sum, 'min': min, 'max': max,
            'abs': abs, 'round': round, 'sorted': sorted, 'enumerate': enumerate,
            'zip': zip, 'map': map, 'filter': filter, 'list': list, 'dict': dict,
            'set': set, 'tuple': tuple, 'int': int, 'float': float, 'str': str,
            'bool': bool, 'print': print, 'isinstance': isinstance, 'type': type,
        }}
        
        # Allow math module
        import math
        namespace['math'] = math
        
        # Capture stdout
        import io
        import contextlib
        f = io.StringIO()
        with contextlib.redirect_stdout(f):
            exec(code, namespace)
        
        output = f.getvalue()
        # Also try to get the last expression value
        if not output:
            try:
                tree = ast.parse(code)
                if tree.body and isinstance(tree.body[-1], ast.Expr):
                    last_expr = ast.get_source_segment(code, tree.body[-1])
                    if last_expr:
                        result = eval(last_expr, namespace)
                        output = str(result)
            except Exception:
                pass
        
        return output.strip(), None
    except Exception as e:
        return None, str(e)


def tool_augmented_reasoning(question, max_rounds=3):
    """
    Reasoning with Python execution tool.
    The model can write code to verify or compute steps.
    """
    messages = [{
        "role": "system",
        "content": """You are a careful problem solver. You can use Python code to verify 
your reasoning or perform calculations. When you need to compute something, 
write Python code between ```python and ``` markers. The code will be executed 
and you'll see the output. Use print() to display results.

Solve problems step by step, using code to verify each step."""
    }, {
        "role": "user",
        "content": question
    }]
    
    full_trace = []
    
    for round_num in range(max_rounds):
        response = chat_openai(messages, temperature=0.0, max_tokens=1000)
        full_trace.append(f"=== Round {round_num + 1} ===")
        full_trace.append(response)
        
        # Check for code blocks
        code_blocks = re.findall(r'```python\n(.*?)```', response, re.DOTALL)
        
        if not code_blocks:
            # No more code to execute — model has its answer
            break
        
        # Execute each code block
        execution_results = []
        for code in code_blocks:
            output, error = safe_execute_python(code.strip())
            if error:
                execution_results.append(f"Error: {error}")
            else:
                execution_results.append(f"Output: {output}")
        
        # Feed results back to model
        result_text = "\n".join(execution_results)
        messages.append({"role": "assistant", "content": response})
        messages.append({
            "role": "user",
            "content": f"Code execution results:\n{result_text}\n\nContinue your reasoning based on these results."
        })
        full_trace.append(f"[Code output: {result_text}]")
    
    return response, full_trace

# Test tool-augmented reasoning
question = (
    "A ball is dropped from a height of 100 meters. Each time it bounces, it reaches "
    "75% of its previous height. What is the total distance the ball travels before "
    "it stops (theoretically, counting both up and down movements)?"
)

answer, trace = tool_augmented_reasoning(question)
print("Reasoning trace:")
for t in trace:
    print(t[:200])
    print()

In [ ]:
# Compare: with and without tool use
hard_math = "What is the sum of all prime numbers less than 50?"

# Without tools
no_tool = ask(f"{hard_math}\nThink step by step.")
print("WITHOUT tool augmentation:")
print(f"  {no_tool[:200]}")

# With tools
with_tool, _ = tool_augmented_reasoning(hard_math)
print(f"\nWITH tool augmentation:")
print(f"  {with_tool[:200]}")

print(f"\nExpected: 328 (2+3+5+7+11+13+17+19+23+29+31+37+41+43+47)")

---
## 5. Reasoning with Claude — Extended Thinking API

Anthropic's Claude models support **extended thinking** — a native thinking mode where Claude generates an internal reasoning trace before responding. This is similar to o1's thinking tokens but with explicit API control.

### Key API parameters:
- `thinking.type = "enabled"` — turns on extended thinking
- `thinking.budget_tokens` — controls how many tokens Claude can use for internal reasoning
- The thinking content is returned as a separate content block

In [ ]:
def claude_with_thinking(question, budget_tokens=5000, model="claude-sonnet-4-20250514"):
    """
    Use Claude's extended thinking API.
    Returns (thinking_text, answer_text, usage).
    """
    response = anthropic_client.messages.create(
        model=model,
        max_tokens=16000,
        thinking={
            "type": "enabled",
            "budget_tokens": budget_tokens,
        },
        messages=[{
            "role": "user",
            "content": question,
        }],
    )
    
    thinking_text = ""
    answer_text = ""
    
    for block in response.content:
        if block.type == "thinking":
            thinking_text = block.thinking
        elif block.type == "text":
            answer_text = block.text
    
    usage = {
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }
    
    return thinking_text, answer_text, usage

# Test extended thinking on a challenging problem
problem = (
    "Prove that for any positive integer n, the sum 1 + 2 + 3 + ... + n = n(n+1)/2. "
    "Use mathematical induction."
)

try:
    thinking, answer, usage = claude_with_thinking(problem, budget_tokens=4000)
    
    print(f"THINKING ({len(thinking.split())} words):")
    print(f"{thinking[:500]}...")
    print(f"\nANSWER:")
    print(f"{answer[:500]}")
    print(f"\nToken usage: {usage}")
except Exception as e:
    print(f"Error (API key may not be set): {e}")
    print("\nTo use Claude's extended thinking:")
    print("  export ANTHROPIC_API_KEY=your-key-here")

In [ ]:
# Compare thinking budgets
budgets_to_test = [1024, 2048, 5000, 10000]
complex_problem = (
    "A group of 100 pirates find 1000 gold coins. They decide to distribute them "
    "using a voting system: the most senior pirate proposes a distribution. "
    "All pirates vote, and if at least 50% approve, the distribution is accepted. "
    "Otherwise, the proposer is thrown overboard and the next most senior proposes. "
    "Pirates are perfectly rational, prefer more gold, and prefer survival. "
    "What distribution does the most senior pirate propose?"
)

budget_data = []

for budget in budgets_to_test:
    try:
        start = time.time()
        thinking, answer, usage = claude_with_thinking(complex_problem, budget_tokens=budget)
        elapsed = time.time() - start
        
        budget_data.append({
            'budget': budget,
            'thinking_words': len(thinking.split()),
            'answer_words': len(answer.split()),
            'time': elapsed,
            'total_tokens': usage['input_tokens'] + usage['output_tokens'],
        })
        print(f"  Budget={budget:5d}: thinking={len(thinking.split()):4d} words, time={elapsed:.1f}s")
    except Exception as e:
        print(f"  Budget={budget}: Error — {e}")
        break

if budget_data:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    b = [d['budget'] for d in budget_data]
    tw = [d['thinking_words'] for d in budget_data]
    tt = [d['time'] for d in budget_data]
    
    ax1.plot(b, tw, 'o-', color='#3498db', linewidth=2)
    ax1.set_xlabel('Budget tokens')
    ax1.set_ylabel('Thinking words generated')
    ax1.set_title('Thinking Depth vs Budget')
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(b, tt, 's-', color='#e74c3c', linewidth=2)
    ax2.set_xlabel('Budget tokens')
    ax2.set_ylabel('Response time (seconds)')
    ax2.set_title('Latency vs Budget')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

### Best practices for extended thinking with Claude

1. **Budget sizing:** Start with 4000-8000 tokens for moderate problems. Use 10000+ for very complex reasoning.

2. **Don't tell Claude how to think:** The thinking process is optimized internally. Prompts like "think step by step" are redundant with extended thinking enabled.

3. **Use for hard problems only:** Simple tasks don't benefit from extended thinking. The overhead isn't worth it for straightforward Q&A.

4. **Streaming:** Extended thinking supports streaming — you can get the thinking content as it's generated, then the final answer.

5. **The thinking content is redacted:** You get a summary, not the raw thinking tokens. This is by design for safety.

---
## Summary

| Technique | Key Idea | Best For |
|-----------|---------|----------|
| **Thinking tokens (o1/o3)** | Hidden extended reasoning before answer | Complex reasoning, math, code |
| **Reasoning trace analysis** | Identify patterns: backtracking, verification, decomposition | Debugging and improving prompts |
| **Tree of Thought (BFS)** | Explore multiple paths, keep best at each depth | Problems with many possible approaches |
| **Tree of Thought (DFS)** | Follow each path deeply, prune bad ones | Independent solution paths |
| **Tool-augmented reasoning** | Execute code for precise computation | Math, data analysis, verification |
| **Claude extended thinking** | Budget-controlled native thinking | Hard problems with Anthropic's API |

**Next:** `08-reasoning-models/expert.ipynb` — process reward models, MCTS, training reasoning models